# Regression — Solutions

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/solutions/07_Regression_solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ~75 min · Day 3

**Important:** Try the exercises in `demos/07_Regression.ipynb` yourself first.

Reference solutions with short explanations — your own version may differ and still be correct.

In [ ]:
# === COURSE SETUP — run this cell first! ===
%pip install -q numpy pandas matplotlib seaborn scikit-learn statsmodels

import os, urllib.request
DATA_URL = "https://raw.githubusercontent.com/LuWidme/uk259/rework/datasets/"
for folder in ("datasets", os.path.join("..", "datasets")):
    os.makedirs(folder, exist_ok=True)
    for fname in ['titanic.csv', 'melb_data.csv', 'Company_data.csv']:
        path = os.path.join(folder, fname)
        if not os.path.exists(path):
            urllib.request.urlretrieve(DATA_URL + fname, path)
print("Setup complete — you are ready to go!")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
penguins = sns.load_dataset('penguins')
tips = sns.load_dataset('tips')
melb = pd.read_csv('../datasets/melb_data.csv')
print('Ready.')

### Task 1: Visualize a Linear Relationship

In [ ]:
g = sns.lmplot(data=penguins, x='bill_length_mm', y='bill_depth_mm')
plt.show()
print('Across all species the line looks flat/negative — see Simpson\'s paradox in the demo.')

## Part 6: Exercises

### Exercise 1: Melbourne Housing Price Prediction

In [ ]:
feature_cols = ['Rooms', 'Distance', 'Bathroom', 'Landsize', 'BuildingArea']
melb_clean = melb.dropna(subset=feature_cols + ['Price'])
X_melb = melb_clean[feature_cols]
y_melb = melb_clean['Price']
X_train, X_test, y_train, y_test = train_test_split(
    X_melb, y_melb, test_size=0.2, random_state=42)
print('Train/test:', X_train.shape, X_test.shape)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f'MAE:  ${mae:,.0f}')
print(f'RMSE: ${rmse:,.0f}')
print(f'R2:   {r2:.3f}')
print('\nCoefficients:')
for name, coef in zip(feature_cols, model.coef_):
    print(f'  {name:>13}: {coef:,.0f}')

In [ ]:
residuals = y_test - y_pred
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_pred, residuals, alpha=0.4)
ax[0].axhline(0, color='red'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Residual')
ax[0].set_title('Residuals vs Predicted')
ax[1].scatter(y_test, y_pred, alpha=0.4)
ax[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax[1].set_xlabel('Actual'); ax[1].set_ylabel('Predicted'); ax[1].set_title('Actual vs Predicted')
plt.tight_layout(); plt.show()

### Bonus Exercise: Feature Selection with Statsmodels

In [ ]:
from statsmodels.formula.api import ols
full_formula = 'tip ~ total_bill + size + smoker + sex + day + time'
result_full = ols(formula=full_formula, data=tips).fit()
print(result_full.summary())

# Only total_bill and size tend to have p < 0.05 -> keep those
result_simple = ols(formula='tip ~ total_bill + size', data=tips).fit()
print(f"\nFull R2:      {result_full.rsquared:.3f}")
print(f"Simplified R2: {result_simple.rsquared:.3f}")
print('Dropping non-significant features barely changes R2 — the simpler model is preferable.')